# Daily product demand inference



This notebook simulates the inference portion of the pipeline: obtain recent sales as JSON, reconstruct the training features, load the trained XGBoost model, predict the incoming day, and send the results to another API endpoint.



Simulation mode is enabled by default. It converts the bundled CSV history to the same JSON contract and prints the outbound request instead of making network calls.

## 1. Configure runtime and API parameters



Databricks job parameters are exposed as widgets. Credentials are read from Databricks Secrets only when real API mode is enabled.

In [1]:
import os

from pathlib import Path

import joblib

import numpy as np

import pandas as pd

from xgboost import XGBRegressor


from config import CONFIG
from correlation_id import generate_correlation_id
from http_session import create_http_session

correlation_id = generate_correlation_id()

http = create_http_session(correlation_id)
display(CONFIG)

print("Simulation mode:", CONFIG.simulation_mode)
print("Requested history days:", CONFIG.history_days)

ImportError: cannot import name 'create_http_session' from 'http_session' (/home/mc/workspace/ml-example-deployment/src/inference/http_session.py)

## 2. Load the trained model contract



Load the XGBoost model and metadata written by the training notebook. Metadata controls category names and exact feature ordering.

In [ ]:
working_directory = Path.cwd().resolve()

artifact_directories = [working_directory / "outputs", working_directory.parent.parent / "outputs"]

ARTIFACT_DIR = next((path for path in artifact_directories if path.exists()), None)

if ARTIFACT_DIR is None:

    raise FileNotFoundError("Could not find outputs/. Run training or deploy the model artifacts first.")



model_path = ARTIFACT_DIR / "xgb_daily_product_demand.json"

metadata_path = ARTIFACT_DIR / "forecast_metadata.joblib"

if not model_path.exists() or not metadata_path.exists():

    raise FileNotFoundError("The trained model or forecast metadata is missing from outputs/.")



artifacts = joblib.load(metadata_path)

configuration = artifacts["configuration"]

DATE_COLUMN = configuration["date_column"]

CATEGORY_COLUMN = configuration["category_column"]

TARGET_COLUMN = configuration["target_column"]

MODEL_COLUMNS = artifacts["model_feature_columns"]

KNOWN_CATEGORIES = artifacts["categories"]



model = XGBRegressor()

model.load_model(model_path)

print(f"Loaded model contract with {len(MODEL_COLUMNS)} features and {len(KNOWN_CATEGORIES)} categories.")

## 3. Query the recent-sales endpoint



Real mode sends an authenticated GET request and expects either a JSON list or an object containing `records` or `data`. Simulation mode creates the same payload from the latest bundled CSV records. The endpoint must supply at least 28 calendar days of history.

In [ ]:
def get_api_token():

    if CONFIG.simulation_mode:
        return "simulation-token"

    try:

        return dbutils.secrets.get(

            scope=CONFIG.secret_scope,

            key=CONFIG.secret_key,

        )

    except NameError as error:

        raise RuntimeError("Real API mode requires Databricks Secrets.") from error





api_token = get_api_token()

request_headers = {

    "Authorization": f"Bearer {api_token}",

    "Accept": "application/json",

    "Content-Type": "application/json",

}



if CONFIG.simulation_mode:

    data_directories = [working_directory / "data", working_directory.parent.parent / "data"]

    data_directory = next((path for path in data_directories if path.exists()), None)

    if data_directory is None:

        raise FileNotFoundError("Simulation mode requires the bundled data directory.")

    source_frames = [pd.read_csv(path) for path in sorted(data_directory.glob("*.csv"))]

    simulated_sales = pd.concat(source_frames, ignore_index=True)

    simulated_sales[DATE_COLUMN] = pd.to_datetime(simulated_sales[DATE_COLUMN], errors="coerce")

    latest_date = simulated_sales[DATE_COLUMN].max()

    recent_sales = simulated_sales.loc[

        simulated_sales[DATE_COLUMN] >= latest_date - pd.Timedelta(days=CONFIG.history_days - 1),

        [DATE_COLUMN, CATEGORY_COLUMN, TARGET_COLUMN],

    ].copy()

    recent_sales[DATE_COLUMN] = recent_sales[DATE_COLUMN].dt.strftime("%Y-%m-%d")

    source_payload = {"records": recent_sales.to_dict(orient="records")}

    print(f"Simulated GET returned {len(source_payload['records']):,} JSON records.")

else:

    response = http.get(

        CONFIG.source_endpoint_url,

        headers=request_headers,

        params={"history_days": CONFIG.history_days},

    )

    response.raise_for_status()

    source_payload = response.json()

    print("Recent-sales GET status:", response.status_code)

## 4. Validate and preprocess the JSON response



Normalize the response into one daily row per known category, fill absent date-category combinations with zero, and verify enough history exists for the model's 28-day features.

In [ ]:
if isinstance(source_payload, list):

    source_records = source_payload

elif isinstance(source_payload, dict):

    source_records = source_payload.get("records", source_payload.get("data"))

else:

    source_records = None



if not isinstance(source_records, list) or not source_records:

    raise ValueError("The recent-sales endpoint must return a non-empty JSON record list.")



recent_sales = pd.DataFrame.from_records(source_records)

required_columns = {DATE_COLUMN, CATEGORY_COLUMN, TARGET_COLUMN}

missing_columns = required_columns.difference(recent_sales.columns)

if missing_columns:

    raise ValueError(f"Recent-sales JSON is missing columns: {sorted(missing_columns)}")



recent_sales[DATE_COLUMN] = pd.to_datetime(recent_sales[DATE_COLUMN], errors="coerce")

recent_sales[CATEGORY_COLUMN] = recent_sales[CATEGORY_COLUMN].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

recent_sales[TARGET_COLUMN] = pd.to_numeric(recent_sales[TARGET_COLUMN], errors="coerce")

valid_rows = (

    recent_sales[DATE_COLUMN].notna()

    & recent_sales[CATEGORY_COLUMN].isin(KNOWN_CATEGORIES)

    & recent_sales[TARGET_COLUMN].notna()

    & recent_sales[TARGET_COLUMN].ge(0)

)

recent_sales = recent_sales.loc[valid_rows, [DATE_COLUMN, CATEGORY_COLUMN, TARGET_COLUMN]]

daily_sales = recent_sales.groupby([DATE_COLUMN, CATEGORY_COLUMN], as_index=False)[TARGET_COLUMN].sum()



latest_date = daily_sales[DATE_COLUMN].max()

first_required_date = latest_date - pd.Timedelta(days=27)

if daily_sales[DATE_COLUMN].min() > first_required_date:

    raise ValueError("At least 28 consecutive calendar days of history are required for inference.")



all_dates = pd.date_range(daily_sales[DATE_COLUMN].min(), latest_date, freq="D")

complete_index = pd.MultiIndex.from_product(

    [all_dates, KNOWN_CATEGORIES], names=[DATE_COLUMN, CATEGORY_COLUMN]

)

daily_sales = (

    daily_sales.set_index([DATE_COLUMN, CATEGORY_COLUMN])

    .reindex(complete_index, fill_value=0)

    .reset_index()

    .sort_values([CATEGORY_COLUMN, DATE_COLUMN])

)

print(f"Prepared {len(all_dates)} days through {latest_date.date()} for {len(KNOWN_CATEGORIES)} categories.")

## 5. Reconstruct features and run inference



Append the incoming day and calculate calendar, lag, and rolling features exactly as in training. Align the encoded columns to the stored model contract before predicting.

In [ ]:
def create_time_features(data):

    featured = data.sort_values([CATEGORY_COLUMN, DATE_COLUMN]).copy()

    featured["Day_Of_Week"] = featured[DATE_COLUMN].dt.dayofweek

    featured["Month"] = featured[DATE_COLUMN].dt.month

    featured["Day_Of_Month"] = featured[DATE_COLUMN].dt.day

    featured["Day_Of_Year"] = featured[DATE_COLUMN].dt.dayofyear

    featured["Is_Weekend"] = featured["Day_Of_Week"].isin([5, 6]).astype(int)

    grouped_sales = featured.groupby(CATEGORY_COLUMN)[TARGET_COLUMN]

    for lag in [1, 7, 14, 28]:

        featured[f"Lag_{lag}"] = grouped_sales.shift(lag)

    for window in [7, 14, 28]:

        featured[f"Rolling_Mean_{window}"] = grouped_sales.transform(

            lambda values: values.shift(1).rolling(window).mean()

        )

        featured[f"Rolling_Std_{window}"] = grouped_sales.transform(

            lambda values: values.shift(1).rolling(window).std()

        )

    return featured





forecast_date = latest_date + pd.Timedelta(days=1)

future_rows = pd.DataFrame({

    DATE_COLUMN: forecast_date,

    CATEGORY_COLUMN: KNOWN_CATEGORIES,

    TARGET_COLUMN: 0.0,

})

history_and_future = pd.concat([daily_sales, future_rows], ignore_index=True)

future_features = create_time_features(history_and_future)

future_features = future_features.loc[future_features[DATE_COLUMN] == forecast_date].copy()



feature_columns = artifacts["raw_feature_columns"]

X_future = pd.get_dummies(future_features[feature_columns], columns=[CATEGORY_COLUMN], dtype=int)

X_future = X_future.reindex(columns=MODEL_COLUMNS, fill_value=0)

if X_future.isna().any().any():

    raise ValueError("Recent sales did not provide enough history to calculate every model feature.")



predicted_quantities = np.rint(np.clip(model.predict(X_future), 0, None)).astype(int)

forecast = future_features[[DATE_COLUMN, CATEGORY_COLUMN]].copy()

forecast["Predicted_Qty"] = predicted_quantities

forecast = forecast.sort_values(CATEGORY_COLUMN).reset_index(drop=True)

display(forecast)

print("Forecast total units:", forecast["Predicted_Qty"].sum())

## 6. Return predictions to the result endpoint



Serialize the forecast as JSON and POST it in real mode. Simulation mode displays the request body without contacting an external service.

In [ ]:
result_payload = {

    "forecast_date": forecast_date.strftime("%Y-%m-%d"),

    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),

    "predictions": [

        {

            "category": row[CATEGORY_COLUMN],

            "predicted_quantity": int(row["Predicted_Qty"]),

        }

        for _, row in forecast.iterrows()

    ],

}



if CONFIG.simulation_mode:

    outbound_result = {

        "simulation": True,

        "url": CONFIG.result_endpoint_url,

        "payload": result_payload,

    }

    print(f"Simulated POST with {len(result_payload['predictions'])} predictions.")

    display(pd.DataFrame(result_payload["predictions"]))

else:

    response = http.post(

        CONFIG.result_endpoint_url,

        headers=request_headers,

        json=result_payload,
    )

    response.raise_for_status()

    try:

        response_body = response.json()

    except requests.exceptions.JSONDecodeError:

        response_body = response.text

    outbound_result = {

        "status_code": response.status_code,

        "response": response_body,

    }

    print("Forecast POST status:", response.status_code)



outbound_result